In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

df_copy = gsl_monthly_nosnow.copy()
df = df_copy.sort_values("date").reset_index(drop=True)

lags = 3

cols_to_lag = [
    "elevation_ft",
    "precip_acft",
    # "snow_water_equiv_in",
    "total_inflow_cfs_acft",
    "evaporation_acft"
]

for col in cols_to_lag:
    for i in range(1, lags+1):
        df[f"{col}_lag{i}"] = df[col].shift(i)

# df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
# df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

df["target_elevation"] = df["elevation_ft"].shift(-1)

df_model = df.dropna()

feature_cols = (
    [c for c in df_model.columns if "lag" in c]
    # + ["month_sin", "month_cos"]
)

X = df_model[feature_cols]
y = df_model["target_elevation"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=4,
        weights="distance"
    ))
])

model.fit(X, y)

future_predictions = []

state = df_model.iloc[-1:].copy()

current_month = int(state["month"].iloc[0])

for step in range(10*12):

    # predict
    X_state = state[feature_cols]
    pred = model.predict(X_state)[0]
    future_predictions.append(pred)

    # update elevation
    state["elevation_ft"] = pred

    # advance month
    current_month = current_month % 12 + 1
    state["month"] = current_month
    # state["month_sin"] = np.sin(2*np.pi*current_month/12)
    # state["month_cos"] = np.cos(2*np.pi*current_month/12)

    # shift lag variables forward
    for col in cols_to_lag:
        for i in range(lags, 1, -1):
            state[f"{col}_lag{i}"] = state[f"{col}_lag{i-1}"]

    # insert newest values
    state["elevation_ft_lag1"] = pred

# future_predictions
last_date = df["date"].iloc[-1]

future_dates = pd.date_range(
    start=last_date + pd.offsets.MonthBegin(1),
    periods=len(future_predictions),
    freq="MS"   # month start
)

pred_df = pd.DataFrame({
    "date": future_dates,
    "predicted_elevation": future_predictions
})

fig = plt.figure(figsize=(10, 6))
plt.plot(pred_df["date"], pred_df["predicted_elevation"])
plt.show()

# Plot the forecast alongside the historical data
plt.figure(figsize=(10,6))

# Plot historical data
plt.plot(
    df["date"],
    df["elevation_ft"],
    label="Observed Elevation"
)

# Plot the forecast from the analysis
plt.plot(
    pred_df["date"],
    pred_df["predicted_elevation"],
    linestyle="--",
    label="kNN Forecast"
)

# Add vertical line separating history & forecast
plt.axvline(last_date, linestyle=":", linewidth=2)
plt.xlabel("Date")
plt.ylabel("Elevation (ft)")
plt.title("Reservoir Elevation Forecast (kNN)")
plt.legend()
plt.show()

In [ ]:
# ==========================================
# ITERATIVE ANALOG FUTURE FORECAST
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

df = gsl_monthly_nosnow.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

features = [
    "elevation_ft",
    "precip_acft",
    "total_inflow_cfs_acft",
    "evaporation_acft"
]

k = 20
years_to_simulate = [2026, 2027, 2028, 2029, 2030]    # <- forecast 5 future years
months_per_step = 12

# -------------------------------------------------
# START FROM LAST OBSERVED STATE
# -------------------------------------------------
simulation_df = df.copy()

current_state = simulation_df.iloc[-1].copy()

future_records = []

# -------------------------------------------------
# ITERATE INTO THE FUTURE
# -------------------------------------------------
for yr in years_to_simulate:  #range(years_to_simulate):

    forecast_month = int(current_state["month"])

    # same calendar month analog search
    historical_states = simulation_df[
        simulation_df["month"] == forecast_month
    ]

    X_hist = historical_states[features]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_hist)

    current_scaled = scaler.transform(
        current_state[features].values.reshape(1, -1)
    )

    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(X_scaled)

    _, idx = knn.kneighbors(current_scaled)

    analog_years = historical_states.iloc[idx[0]]["year"].values

    # -----------------------------------------
    # BUILD ANALOG ENSEMBLE FUTURE
    # -----------------------------------------
    future_curves = []

    for ay in analog_years:

        future = df[
            (df["year"] == ay) &
            (df["month"] >= forecast_month)
        ][["date","elevation_ft"]].head(months_per_step)

        future_curves.append(
            future.set_index("date")
        )

    ensemble = pd.concat(future_curves, axis=1)

    median_future = ensemble.median(axis=1)

    # -----------------------------------------
    # CREATE SYNTHETIC FUTURE YEAR
    # -----------------------------------------
    new_year = []

    for i, (date, elev) in enumerate(median_future.items()):

        new_row = current_state.copy()

        new_row["date"] = simulation_df["date"].iloc[-1] + pd.DateOffset(months=i+1)
        new_row["year"] = new_row["date"].year
        new_row["month"] = new_row["date"].month
        new_row["elevation_ft"] = elev

        new_year.append(new_row)

    new_year_df = pd.DataFrame(new_year)

    # append synthetic future to history
    simulation_df = pd.concat(
        [simulation_df, new_year_df],
        ignore_index=True
    )

    # update state to end of simulated year
    current_state = simulation_df.iloc[-1]

    future_records.append(new_year_df)

# -------------------------------------------------
# COMBINE FUTURE FORECAST
# -------------------------------------------------
future_forecast = pd.concat(future_records)

# -------------------------------------------------
# PLOT RESULTS
# -------------------------------------------------
plt.figure(figsize=(12,6))

subset = df[df["year"] >= 2019]
plt.plot(
    subset["date"],
    subset["elevation_ft"],
    label="Observed"
)

plt.plot(
    future_forecast["date"],
    future_forecast["elevation_ft"],
    linestyle="--",
    linewidth=3,
    label="Analog Future Simulation"
)

plt.axvline(df["date"].iloc[-1], linestyle=":")

plt.ylabel("Elevation (ft)")
plt.xlabel("Date")
plt.title("Multi-Year Analog Ensemble Forecast")
plt.legend()

plt.show()

df.columns

In [ ]:
# ==============================
# Analog Ensemble Reservoir Forecast
# ==============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# -------------------------------------------------
# 1. PREP DATA
# -------------------------------------------------
df = gsl_monthly_nosnow.copy()

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# -------------------------------------------------
# 2. DEFINE FORECAST ORIGIN (AUTOMATIC)
# -------------------------------------------------
target_year = df["year"].iloc[-1]
forecast_month = df["month"].iloc[-1]

print("Forecast issued from:")
print("Year:", target_year)
print("Month:", forecast_month)

# -------------------------------------------------
# 3. DEFINE HYDROLOGIC STATE VARIABLES
# -------------------------------------------------
features = [
    "elevation_ft",
    "precip_acft",
    "total_inflow_cfs_acft",
    "evaporation_acft"
]

current_state = df.loc[df.index[-1], features]

# -------------------------------------------------
# 4. BUILD HISTORICAL ANALOG CANDIDATES
#    (ONLY SAME MONTH!)
# -------------------------------------------------
historical_states = df[df["month"] == forecast_month]

# remove current year so it can't match itself
historical_states = historical_states[
    historical_states["year"] != target_year
]

# -------------------------------------------------
# 5. FIND ANALOG YEARS (kNN SEARCH)
# -------------------------------------------------
X_hist = historical_states[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_hist)

current_scaled = scaler.transform(
    current_state.values.reshape(1, -1)
)

k = 20   # typical operational range = 20–50
knn = NearestNeighbors(n_neighbors=k)
knn.fit(X_scaled)

distances, indices = knn.kneighbors(current_scaled)

analog_years = historical_states.iloc[
    indices[0]
]["year"].values

print("Analog years:", analog_years)

# -------------------------------------------------
# 6. EXTRACT FUTURE TRAJECTORIES
# -------------------------------------------------
future_curves = []

for yr in analog_years:

    future = df[
        (df["year"] == yr) &
        (df["month"] >= forecast_month)
    ][["date", "elevation_ft"]]

    future = future.set_index("date")

    future_curves.append(future)

# -------------------------------------------------
# 7. BUILD ANALOG ENSEMBLE
# -------------------------------------------------
ensemble = pd.concat(future_curves, axis=1)

ensemble.columns = [f"analog_{i}" for i in range(len(ensemble.columns))]

median = ensemble.median(axis=1)
p10 = ensemble.quantile(0.10, axis=1)
p90 = ensemble.quantile(0.90, axis=1)

# -------------------------------------------------
# 8. PLOT FORECAST (Water Agency Style)
# -------------------------------------------------
plt.figure(figsize=(12,6))

# Plot historical elevation
plt.plot(
    df["date"],
    df["elevation_ft"],
    label="Observed Elevation",
    linewidth=2
)

# Plot the ensemble median forecast
plt.plot(
    median.index,
    median,
    label="Analog Ensemble Median",
    linewidth=3
)

# Add uncertainty band
plt.fill_between(
    median.index,
    p10,
    p90,
    alpha=0.3,
    label="80% Prediction Range"
)

# Plot forecast start marker
plt.axvline(
    df["date"].iloc[-1],
    linestyle="--",
    linewidth=2,
    label="Forecast Start"
)

plt.xlabel("Date")
plt.ylabel("Elevation (ft)")
plt.title("Analog Ensemble Reservoir Elevation Forecast")
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

# ---------------------------------------------------
# 1. PREP DATA
# ---------------------------------------------------

df = gsl_monthly_nosnow.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

features = [
    "month",
    "total_inflow_cfs_acft",
    "precip_acft",
    "evaporation_acft"
]

target = "elevation_ft"

# starting dataset used for recursive forecasting
df_work = df.copy()

# years you want to forecast
forecast_years = [2025, 2026, 2027, 2028, 2029, 2030]

all_forecasts = []

# ---------------------------------------------------
# 2. ANALOG ENSEMBLE FORECAST LOOP
# ---------------------------------------------------
df_work["forecast"] = False

for target_year in forecast_years:

    yearly_predictions = []

    print(f"Forecasting {target_year}...")

    for month in range(1, 13):

        # ------------------------------
        # TRAIN DATA ONLY WITH PAST YEARS
        # ------------------------------
        # train = df_work[df_work["year"] < target_year]

        train = df_work[
        (df_work["year"] < target_year) &
        (df_work.get("forecast", False) != True)
        ]

        train = train.dropna(subset=features + [target])

        X_train = train[features]
        y_train = train[target]

        # Fit kNN analog finder
        knn = NearestNeighbors(n_neighbors=4)
        knn.fit(X_train)

        # ------------------------------
        # CREATE FORECAST STATE VECTOR
        # ------------------------------
        # use most recent known conditions
        # last_obs = df_work[df_work["month"] == month].iloc[-1]
        last_obs = df_work[
        (df_work["month"] == month) &
        (df_work.get("forecast", False) != True)
        ].iloc[-1]

        X_target = last_obs[features].values.reshape(1, -1)

        # ------------------------------
        # FIND ANALOG YEARS
        # ------------------------------
        distances, indices = knn.kneighbors(X_target)

        analogs = train.iloc[indices[0]]

        # Analog ensemble mean
        predicted_elevation = analogs[target].mean()

        yearly_predictions.append({
            "year": target_year,
            "month": month,
            "date": pd.Timestamp(f"{target_year}-{month:02d}-01"),
            "elevation_ft": predicted_elevation,
            "forecast": True
        })

    # ------------------------------
    # ADD YEAR FORECAST TO HISTORY
    # (this enables recursive future prediction)
    # ------------------------------
    year_df = pd.DataFrame(yearly_predictions)

    df_work = pd.concat([df_work, year_df], ignore_index=True)

    all_forecasts.append(year_df)

# combine all forecast years
future_forecast = pd.concat(all_forecasts).reset_index(drop=True)

print("\nForecast complete.")

# -------------------------------------------------
# PLOT RESULTS
# -------------------------------------------------
plt.figure(figsize=(12,6))

subset = df[df["year"] >= 2019]
plt.plot(
    subset["date"],
    subset["elevation_ft"],
    label="Observed"
)

plt.plot(
    future_forecast["date"],
    future_forecast["elevation_ft"],
    linestyle="--",
    linewidth=3,
    label="Analog Future Simulation"
)

plt.axvline(df["date"].iloc[-1], linestyle=":")

plt.ylabel("Elevation (ft)")
plt.xlabel("Date")
plt.title("Multi-Year Analog Ensemble Forecast")
plt.legend()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

df = gsl_monthly_nosnow.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

years_to_simulate = 15 # number of years to simulate into the future
n_analogs = 4 # number of nearest neighbors to consider as analogs
n_blend = 3 # number of analogs to blend together for each step
noise_scale = 0.03 # 3% of the standard deviation of each feature

state_features = [
    # "snow_water_equiv_in",
    "precip_acft",
    "total_inflow_cfs_acft",
    "evaporation_acft",
    "elevation_ft"
]

## Create the yearly states ##
year_features = (
    df.groupby("year").agg({
            # "snow_water_equiv_in": "mean",
            "precip_acft": "sum",
            "total_inflow_cfs_acft": "sum",
            "evaporation_acft": "sum",
            "elevation_ft": "mean"
        })
        .dropna()
)

current_year = year_features.index.max()
initial_state = year_features.loc[[current_year]].copy()

prediction_dfs = []

for i in range(101):

    # Reset the start state for each simulation run
    current_state = initial_state.copy()

    future_trajectory = []
    historical_states = year_features.iloc[:-1].copy()

    for step in range(years_to_simulate):

        # Scale historical states for KNN
        scaler = StandardScaler()
        X_hist = scaler.fit_transform(historical_states)

        # Fit KNN on historical states
        knn = NearestNeighbors(n_neighbors=n_analogs)
        knn.fit(X_hist)

        # Scale the current state using the same scaler
        X_target = scaler.transform(current_state)

        # Get indices of nearest neighbors
        distances, indices = knn.kneighbors(X_target)

        # Get the years corresponding to the nearest neighbors
        analog_years = historical_states.index[indices[0]]
        analog_distances = distances[0]
        print("Analog distances:", analog_distances)

        # chosen_analog = np.random.choice(analog_years, k=2)

        ## Use distance-weighted sampling ##
        weights = 1 / (analog_distances + 1e-6)
        probabilities = weights / weights.sum()
        print("Sampling probabilities:", probabilities)

        # Use the weighted probabilities to randomly select analog years to blend together
        chosen_years = np.random.choice(
            analog_years,
            size=n_blend,
            replace=False,
            p=probabilities
        )

        # Generate random blend weights from a Dirichlet distribution to ensure they sum to 1
        blend_weights = np.random.dirichlet(np.ones(n_blend))

        states = []

        # For each chosen analog year, get the next year's state and weight it by the blend weight
        for yr, w in zip(chosen_years, blend_weights):

            next_year = yr + 1
            if next_year not in year_features.index:
                continue

            states.append(year_features.loc[next_year] * w)

        if len(states) == 0:
            break

        # Blend the states together to get the next state and add random noise to introduce variability
        next_state = pd.DataFrame([sum(states)])
        random_noise = np.random.normal(0, noise_scale * year_features.std(), size=next_state.shape)
        next_state += random_noise
        next_state.index = [current_year + step + 1]
        future_trajectory.append(next_state)

        # Advance the simulation state
        current_state = next_state

    # Combine the future trajectory into a single dataframe and add year and date columns for plotting
    future_df = pd.concat(future_trajectory)
    final_df = future_df.copy()
    final_df["year"] = final_df.index
    final_df["date"] = pd.to_datetime(
        {
            "year": final_df["year"],
            "month": 1,
            "day": 1
        }
    )

    # Store the final dataframe for this simulation run in the list of predictions
    prediction_dfs.append(final_df)

# Plot historical elevation and future projection
plt.figure(figsize=(14,6))

# Create a historical yearly aggregated dataframe for plotting
yearly_historical_data = (
        df.groupby("year")
        .agg({
            # "snow_water_equiv_in": "mean",
            "precip_acft": "sum",
            "total_inflow_cfs_acft": "sum",
            "evaporation_acft": "sum",
            "elevation_ft": "mean"
        })
        .dropna()
)

yearly_historical_data["year"] = yearly_historical_data.index

# Plot historical elevation
plt.plot(
    yearly_historical_data["year"],
    yearly_historical_data["elevation_ft"],
    label="Observed"
)

# Plot future projections for elevation
for prediction in prediction_dfs:
    plt.plot(
        prediction["year"],
        prediction["elevation_ft"],
        "--",
        linewidth=1
    )

plt.legend()
plt.grid(True)
plt.title("Multi-Year Analog Elevation Projection")
plt.xlabel("Year")
plt.ylabel("Elevation (ft)")
plt.show()
